[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/hams-chadi/hams-assessment/blob/main/notebooks/eval_20samples.ipynb
)

# Arabic-to-English Speech Translation — 20-Sample Evaluation

This notebook evaluates three systems on 20 test samples ordered by increasing
audio duration (shortest to longest). No training or preprocessing — just load
the saved model and run.

**Systems compared:**
1. Zero-shot S2TT + Kokoro TTS
2. Fine-tuned S2TT + Kokoro TTS (LoRA best_adapter)
3. Direct S2ST (SeamlessM4Tv2ForSpeechToSpeech)

**What is produced:**
- Audio output for each system on each sample
- Latency per sample and per system
- Latency comparison charts
- Human evaluation template CSV


## 1. Setup

In [ ]:
import os
import json
import time
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import torch
import torchaudio
import librosa
from IPython.display import Audio as IPyAudio, display
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
from peft import PeftModel

# ---- Paths (hardcoded absolute paths for Windows compatibility) ----
PROJECT_ROOT = str(Path("..").resolve())
ADAPTER_DIR   = PROJECT_ROOT + r"\checkpoints\lora_s2tt\best_adapter"
PROC_DIR      = PROJECT_ROOT + r"\checkpoints\lora_s2tt\processor"
RESULTS_DIR   = Path(PROJECT_ROOT) / "results"
AUDIO_DIR     = Path(PROJECT_ROOT) / "audio_outputs"
TEST_SET      = Path(PROJECT_ROOT) / "test_set.json"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL    = "facebook/seamless-m4t-v2-large"
TGT_LANG      = "eng"
SAMPLE_RATE   = 16_000
MAX_LABEL_LEN = 128
EVAL_N        = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Load processor
processor = AutoProcessor.from_pretrained(PROC_DIR)
print("Processor loaded.")

Device : cuda
GPU    : NVIDIA GeForce RTX 5080
VRAM   : 17.1 GB
Processor loaded.


In [4]:
# Load processor (shared by all systems)
processor = AutoProcessor.from_pretrained(str(PROC_DIR))
print("Processor loaded.")

Processor loaded.


## 2. Load Test Samples (Increasing Duration)

In [5]:

# Load test set from disk
if not TEST_SET.exists():
    raise FileNotFoundError(
        f"{TEST_SET} not found.\n"
        "Run this cell in your main notebook first:\n\n"
        "    test_export = []\n"
        "    for i in range(len(ds_test)):\n"
        "        test_export.append({\n"
        '            "audio_path": df_test.iloc[i]["audio_path"],\n'
        '            "sentence"  : ds_test[i]["sentence"],\n'
        '            "translation": ds_test[i]["translation"],\n'
        "        })\n"
        "    import json\n"
        '    with open(ROOT / "test_set.json", "w", encoding="utf-8") as f:\n'
        "        json.dump(test_export, f, ensure_ascii=False, indent=2)\n"
    )

with open(TEST_SET, encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Loaded {len(test_data)} test examples from {TEST_SET}")


Loaded 300 test examples from C:\Users\PC\Desktop\Aymane\hams\arabic_en_translation\test_set.json


In [6]:

def load_audio(path):
    path = str(path)
    # Remap paths saved with 'arabic_en_translation\' prefix
    if path.startswith("arabic_en_translation"):
        path = PROJECT_ROOT + "\\" + path[len("arabic_en_translation"):].lstrip("\\/")
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

# Measure duration of every test example
print("Measuring durations ...")
durations = []
for i, ex in enumerate(test_data):
    try:
        arr = load_audio(ex["audio_path"])
        durations.append((len(arr) / SAMPLE_RATE, i))
    except Exception:
        pass  # skip unreadable files

# Sort by duration, pick EVAL_N evenly spaced
durations.sort(key=lambda x: x[0])
step = max(1, len(durations) // EVAL_N)
selected = [durations[i * step] for i in range(EVAL_N)][:EVAL_N]

eval_indices   = [idx for _, idx in selected]
eval_durations = [dur for dur, _ in selected]

print(f"\nSelected {EVAL_N} samples:")
print(f"  Shortest : {eval_durations[0]:.2f}s")
print(f"  Longest  : {eval_durations[-1]:.2f}s")
print(f"  Durations: {[round(d,2) for d in eval_durations]}")


Measuring durations ...

Selected 20 samples:
  Shortest : 1.90s
  Longest  : 6.24s
  Durations: [1.9, 2.54, 2.81, 2.98, 3.1, 3.22, 3.31, 3.43, 3.5, 3.62, 3.74, 3.89, 3.98, 4.1, 4.18, 4.3, 4.44, 4.82, 5.18, 6.24]


## 3. TTS Helper (Kokoro)

In [7]:

# Load Kokoro TTS once — used by both S2TT systems
TTS_SR = 24000
_kokoro = None

try:
    from kokoro import KPipeline
    _kokoro = KPipeline(lang_code="a")
    print("Kokoro TTS ready.")
except Exception as e:
    print("Kokoro unavailable:", e)
    print("Falling back to pyttsx3.")

def synthesize_speech(text, output_path):
    # Synthesize English speech. Returns (waveform_float32, sample_rate).
    output_path = str(output_path)
    text = (text or "").strip()
    if not text:
        silence = np.zeros(TTS_SR, dtype=np.float32)
        sf.write(output_path, silence, TTS_SR)
        return silence, TTS_SR

    if _kokoro is not None:
        chunks = []
        for _, _, audio in _kokoro(text, voice="af_heart", speed=1.0):
            if audio is not None and len(audio) > 0:
                chunks.append(np.asarray(audio, dtype=np.float32))
        wav = np.concatenate(chunks) if chunks else np.zeros(TTS_SR, dtype=np.float32)
        sf.write(output_path, wav, TTS_SR)
        return wav, TTS_SR

    # pyttsx3 fallback
    import pyttsx3
    engine = pyttsx3.init()
    engine.setProperty("rate", 165)
    engine.save_to_file(text, output_path)
    engine.runAndWait()
    wav, sr = sf.read(output_path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    return wav.astype(np.float32), sr


c:\Users\PC\Desktop\Aymane\hams\hams_venv\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\PC\Desktop\Aymane\hams\hams_venv\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Kokoro TTS ready.


## 4. Pass 1 — Zero-Shot S2S Direct

In [ ]:
# ------------------------------------------------------------
# Pass 1: Zero-shot Direct S2ST (base model, no fine-tuning)
# ------------------------------------------------------------
from transformers import SeamlessM4Tv2ForSpeechToSpeech

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("Pass 1: Loading zero-shot S2ST model ...")
s2st_zs = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    BASE_MODEL, dtype=torch.float32
).to(DEVICE).eval()
print("Zero-shot S2ST model ready.")

results      = {"zero_shot": [], "fine_tuned": [], "stt_kokoro": []}
zs_wavs      = []
zs_latencies = []

for rank, idx in enumerate(eval_indices):
    ex    = test_data[idx]
    audio = load_audio(ex["audio_path"])
    dur   = eval_durations[rank]

    inputs = processor(audios=audio, sampling_rate=SAMPLE_RATE,
                       return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = s2st_zs.generate(**inputs, tgt_lang=TGT_LANG)
    total = round((time.perf_counter() - t0) * 1000, 1)

    if hasattr(out, "waveform"):
        wav = out.waveform
    elif isinstance(out, (list, tuple)):
        wav = out[0]
    else:
        wav = out
    wav = wav.detach().cpu().float().numpy().squeeze()

    sf.write(str(AUDIO_DIR / f"zs_{rank:02d}.wav"), wav, 16000)
    zs_wavs.append((wav, 16000))
    zs_latencies.append(total)
    results["zero_shot"].append({
        "rank": rank + 1, "duration_s": round(dur, 2),
        "arabic": ex["sentence"], "reference": ex["translation"],
        "latency_ms": total,
    })
    print(f"  [{rank+1:02d}] {dur:.2f}s | {total}ms")

del s2st_zs
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("Pass 1 done.")

## 5. Pass 2 — Fine-Tuned S2TT + Kokoro

In [ ]:

print("Loading fine-tuned model ...")
_base = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    BASE_MODEL, dtype=torch.float32
)
_peft = PeftModel.from_pretrained(_base, str(ADAPTER_DIR))
ft_model = _peft.merge_and_unload().to(DEVICE).eval()
print("Fine-tuned model ready.")

ft_wavs      = []
ft_latencies = []

for rank, idx in enumerate(eval_indices):
    ex    = test_data[idx]
    audio = load_audio(ex["audio_path"])
    dur   = eval_durations[rank]

    # S2TT
    t0 = time.perf_counter()
    inputs = processor(audios=audio, sampling_rate=SAMPLE_RATE,
                       return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        tokens = ft_model.generate(**inputs, tgt_lang=TGT_LANG,
                                   num_beams=5, max_new_tokens=MAX_LABEL_LEN)
    text   = processor.batch_decode(tokens, skip_special_tokens=True)[0]
    t_s2tt = (time.perf_counter() - t0) * 1000

    # TTS
    t0 = time.perf_counter()
    wav, tts_sr = synthesize_speech(text, AUDIO_DIR / f"ft_{rank:02d}.wav")
    t_tts = (time.perf_counter() - t0) * 1000

    total = round(t_s2tt + t_tts, 1)
    ft_wavs.append((wav, tts_sr))
    ft_latencies.append(total)
    results["fine_tuned"].append({
        "rank": rank + 1, "duration_s": round(dur, 2),
        "arabic": ex["sentence"], "reference": ex["translation"],
        "translation": text, "latency_ms": total,
        "s2tt_ms": round(t_s2tt, 1), "tts_ms": round(t_tts, 1),
    })
    print(f"  [{rank+1:02d}] {dur:.2f}s | {total}ms | {repr(text[:55])}")

del ft_model, _peft, _base
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("Pass 2 done.")


  [09] 3.50s | 1476.7ms | "I don't care if it snows."
  [10] 3.62s | 1339.8ms | 'Can you speak German?'


## 6. Pass 3 — Direct S2ST (Fine-tuned STT + SeamlessM4T Vocoder)

In [ ]:
# ------------------------------------------------------------
# Pass 3: Fine-tuned S2TT + SeamlessM4T Vocoder (Direct S2ST)
# ------------------------------------------------------------
# This uses the fine-tuned text decoder for better translation
# quality, then the SeamlessM4T built-in vocoder for speech
# generation — combining the best of both approaches.
from transformers import SeamlessM4Tv2ForSpeechToSpeech

print("Loading fine-tuned model for vocoder path ...")
_base2 = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    BASE_MODEL, dtype=torch.float32
)
_peft2 = PeftModel.from_pretrained(_base2, ADAPTER_DIR)
ft_model2 = _peft2.merge_and_unload().to(DEVICE).eval()

print("Loading S2ST model for vocoder ...")
s2st_model = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    BASE_MODEL, dtype=torch.float32
).to(DEVICE).eval()

# Copy fine-tuned shared weights into S2ST model
print("Applying fine-tuned weights to S2ST model ...")
ft_state   = ft_model2.state_dict()
s2st_state = s2st_model.state_dict()
copied = 0
for k in s2st_state:
    if k in ft_state and ft_state[k].shape == s2st_state[k].shape:
        s2st_state[k].copy_(ft_state[k])
        copied += 1
s2st_model.load_state_dict(s2st_state)
print(f"Copied {copied} fine-tuned tensors into S2ST model.")

# Free the S2TT model — no longer needed
del ft_model2, _peft2, _base2
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

s2st_wavs      = []
s2st_latencies = []

for rank, idx in enumerate(eval_indices):
    ex    = test_data[idx]
    audio = load_audio(ex["audio_path"])
    dur   = eval_durations[rank]

    inputs = processor(audios=audio, sampling_rate=SAMPLE_RATE,
                       return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = s2st_model.generate(**inputs, tgt_lang=TGT_LANG)
    total = round((time.perf_counter() - t0) * 1000, 1)

    if hasattr(out, "waveform"):
        wav = out.waveform
    elif isinstance(out, (list, tuple)):
        wav = out[0]
    else:
        wav = out
    wav = wav.detach().cpu().float().numpy().squeeze()

    sf.write(str(AUDIO_DIR / f"s2st_{rank:02d}.wav"), wav, 16000)
    s2st_wavs.append((wav, 16000))
    s2st_latencies.append(total)
    results["stt_kokoro"].append({
        "rank": rank + 1, "duration_s": round(dur, 2),
        "arabic": ex["sentence"], "reference": ex["translation"],
        "latency_ms": total,
    })
    print(f"  [{rank+1:02d}] {dur:.2f}s | {total}ms")

del s2st_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

with open(str(RESULTS_DIR / "20sample_eval.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Pass 3 done. Results saved.")

## 7. Audio Players and Latency per Sample

In [ ]:
print("=" * 70)
print("20-SAMPLE EVALUATION — Arabic Speech to English Speech")
print("Ordered by increasing audio duration")
print("=" * 70)

for rank in range(EVAL_N):
    idx = eval_indices[rank]
    ex  = test_data[idx]
    dur = eval_durations[rank]
    p1  = results["zero_shot"][rank]
    p2  = results["fine_tuned"][rank]
    p3  = results["stt_kokoro"][rank]

    print(f"\n{'='*70}")
    print(f"Sample {rank+1:02d}  |  Duration: {dur:.2f}s")
    print(f"  Arabic    : {ex['sentence']}")
    print(f"  Reference : {ex['translation']}")
    print()

    print(f"  [1] Zero-shot Direct S2ST  |  Latency: {p1['latency_ms']}ms")
    w, sr = zs_wavs[rank]
    display(IPyAudio(w, rate=sr))

    print(f"  [2] Fine-tuned S2TT + Kokoro  |  Total: {p2['latency_ms']}ms"
          f"  (S2TT {p2['s2tt_ms']}ms + TTS {p2['tts_ms']}ms)")
    print(f"      {repr(p2['translation'])}")
    w, sr = ft_wavs[rank]
    display(IPyAudio(w, rate=sr))

    print(f"  [3] Fine-tuned S2TT + SeamlessM4T Vocoder  |  Latency: {p3['latency_ms']}ms")
    w, sr = s2st_wavs[rank]
    display(IPyAudio(w, rate=sr))

## 8. Latency Comparison Charts

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

x     = list(range(1, EVAL_N + 1))
xlabs = [f"{i}\n({round(d,1)}s)" for i, d in zip(x, eval_durations)]

axes[0].plot(x, zs_latencies,    marker="o", label="Zero-shot Direct S2ST",
             color="lightcoral",  linewidth=1.5)
axes[0].plot(x, ft_latencies,    marker="s", label="Fine-tuned S2TT + Kokoro",
             color="seagreen",    linewidth=1.5)
axes[0].plot(x, s2st_latencies,  marker="^", label="Fine-tuned S2TT + SeamlessM4T Vocoder",
             color="steelblue",   linewidth=1.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(xlabs, fontsize=7)
axes[0].set_xlabel("Sample (audio duration)")
axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("End-to-End Latency — 3 Systems (20 samples, increasing duration)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(eval_durations, zs_latencies,   label="Zero-shot Direct S2ST",
                color="lightcoral", s=70, zorder=3)
axes[1].scatter(eval_durations, ft_latencies,   label="Fine-tuned S2TT + Kokoro",
                color="seagreen",   s=70, zorder=3)
axes[1].scatter(eval_durations, s2st_latencies, label="Fine-tuned S2TT + SeamlessM4T Vocoder",
                color="steelblue",  s=70, zorder=3)
axes[1].set_xlabel("Audio duration (s)")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency vs Audio Duration")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "20sample_latency.png"), dpi=150)
plt.show()
print("Saved results/20sample_latency.png")

## 9. Summary Table

In [ ]:
summary = pd.DataFrame({
    "System": [
        "Zero-shot Direct S2ST",
        "Fine-tuned S2TT + Kokoro",
        "Fine-tuned S2TT + SeamlessM4T Vocoder",
    ],
    "Mean (ms)": [
        round(np.mean(zs_latencies),    1),
        round(np.mean(ft_latencies),    1),
        round(np.mean(s2st_latencies),  1),
    ],
    "p50 (ms)": [
        round(np.percentile(zs_latencies,   50), 1),
        round(np.percentile(ft_latencies,   50), 1),
        round(np.percentile(s2st_latencies, 50), 1),
    ],
    "p95 (ms)": [
        round(np.percentile(zs_latencies,   95), 1),
        round(np.percentile(ft_latencies,   95), 1),
        round(np.percentile(s2st_latencies, 95), 1),
    ],
    "Max (ms)": [
        round(np.max(zs_latencies),    1),
        round(np.max(ft_latencies),    1),
        round(np.max(s2st_latencies),  1),
    ],
})
summary.to_csv(str(RESULTS_DIR / "20sample_summary.csv"), index=False)
print("Latency Summary — 20 samples, increasing duration")
print("=" * 60)
print(summary.to_string(index=False))

## 10. Human Evaluation Template

In [ ]:

# Export a CSV for a human reviewer to fill in the 4 rating columns.
# The translations and latencies are pre-filled.

rows = []
for rank in range(EVAL_N):
    idx = eval_indices[rank]
    ex  = test_data[idx]
    rows.append({
        "sample_id"                    : rank + 1,
        "duration_s"                   : round(eval_durations[rank], 2),
        "arabic_source"                : ex["sentence"],
        "english_reference"            : ex["translation"],
        "zeroshot_translation"         : results["zero_shot"][rank]["translation"],
        "finetuned_translation"        : results["fine_tuned"][rank]["translation"],
        "zeroshot_latency_ms"          : zs_latencies[rank],
        "finetuned_latency_ms"         : ft_latencies[rank],
        "s2st_latency_ms"              : s2st_latencies[rank],
        "meaning_preservation_1to5"    : "",
        "fluency_1to5"                 : "",
        "completeness_1to5"            : "",
        "serious_mistranslation_yes_no": "",
        "notes"                        : "",
    })

human_df = pd.DataFrame(rows)
human_df.to_csv(
    RESULTS_DIR / "human_eval_20samples.csv",
    index=False, encoding="utf-8-sig",
)
print("Human evaluation template saved to results/human_eval_20samples.csv")
print("Fill in the last 4 columns for each sample.")
print()
print(human_df[["sample_id", "duration_s",
                "arabic_source", "finetuned_translation"]].to_string(index=False))


In [ ]:
# ------------------------------------------------------------
# Generate English audio for human evaluation using Kokoro
# Run this from inside arabic_en_translation/
# ------------------------------------------------------------
import soundfile as sf
import numpy as np
import pandas as pd
from pathlib import Path

# Load the CSV
df = pd.read_csv(
    Path("..") / "results" / "human_eval_template.csv",
    encoding="utf-8-sig"
)

AUDIO_DIR = Path("..") / "audio_outputs" / "human_eval"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Load Kokoro
from kokoro import KPipeline
_kokoro = KPipeline(lang_code="a")
print("Kokoro ready.")

def synthesize(text, path):
    text = (text or "").strip()
    if not text:
        sf.write(str(path), np.zeros(24000, dtype=np.float32), 24000)
        return
    chunks = []
    for _, _, audio in _kokoro(text, voice="af_heart", speed=1.0):
        if audio is not None and len(audio) > 0:
            chunks.append(np.asarray(audio, dtype=np.float32))
    wav = np.concatenate(chunks) if chunks else np.zeros(24000, dtype=np.float32)
    sf.write(str(path), wav, 24000)

# Generate audio for each sample
print(f"Generating audio for {len(df)} samples ...")
for _, row in df.iterrows():
    sid = int(row["id"])

    # Reference English audio
    ref_path = AUDIO_DIR / f"sample_{sid:02d}_reference.wav"
    synthesize(str(row["english_reference"]), ref_path)

    # Model output audio
    out_path = AUDIO_DIR / f"sample_{sid:02d}_model_output.wav"
    synthesize(str(row["model_output"]), out_path)

    print(f"  [{sid:02d}] reference: {row['english_reference']}")
    print(f"        model   : {row['model_output']}")

print(f"\nAll audio saved to {AUDIO_DIR}")
print("Files:")
for f in sorted(AUDIO_DIR.glob("*.wav")):
    print(f"  {f.name}")

c:\Users\PC\Desktop\Aymane\hams\hams_venv\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\PC\Desktop\Aymane\hams\hams_venv\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Kokoro ready.
Generating audio for 20 samples ...
  [01] reference: Where do you play tennis?
        model   : Where do you play tennis?
  [02] reference: We found the main door locked.
        model   : We found the main door locked.
  [03] reference: I’ll read a book.
        model   : I will read a book.
  [04] reference: I expect to arrive soon.
        model   : I expect to arrive soon.
  [05] reference: It’s a hard question to answer.
        model   : That's a difficult question to answer.
  [06] reference: My parents live in the village.
        model   : My parents live in the village.
  [07] reference: I love American movies.
        model   : I like American movies.
  [08] reference: Do you have a lighter?
        model   : Do you have it or not?
  [09] reference: You were born in this hospital.
        model   : This hospital is where I was born.
  [10] reference: Who watches people will die worried.
        model   : Whoever watched people died.
  [11] reference: I don’t 